# CDA REST API — Direct HTTP Access (no `cdapython`)

Notebooks [02](02_cda_exploration.ipynb) and [03](03_cross_node_analysis.ipynb) use the
[`cdapython`](https://pypi.org/project/cdapython/) client. Everything they do is HTTP under the hood:
CDA is a [FastAPI](https://github.com/CancerDataAggregator/cda-api) service you can call directly with
nothing but an HTTP client.

**Why you might prefer raw HTTP:**

- **Zero install.** No `cdapython`, no `cda-client`, no pinned pandas. Just `requests` (or `curl`, or R).
- **Any language.** R, JavaScript, Julia, shell — the examples at the end show curl and R.
- **Version-independent.** Both `cdapython` 2.0.14 bugs documented in this repo's README are client-side only,
  and neither exists over HTTP: the pandas `StringDtype` crash in `column_values()` (no
  `pd.set_option('future.infer_string', False)` needed here), and the `KeyError:
  'anatomic_site_containing_terms'` that makes `get_file_data()` fail outright against the CDA June 2026
  release. Section 5 below runs that same file query successfully with plain `requests`.
- **Capabilities the pinned client doesn't expose.** `SEARCH_LIST` (ontology keyword search) is in the live
  OpenAPI spec but absent from `cda_client` 2.0.1's request model.
- **Transparency.** Every response includes `query_sql` — the SQL CDA generated for your query.

**What you give up:** DataFrames, automatic pagination, client-side column-name validation, and `cdapython`'s
friendly reshaping of the raw response. This notebook reimplements the parts that matter in ~20 lines.

**No authentication is required for any call here.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedorov/idc-cda/blob/main/notebooks/04_cda_rest_api.ipynb)

In [1]:
%pip install --upgrade -q requests pandas

/Users/af61/github/idc-cda/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd

BASE = "https://cda.datacommons.cancer.gov"   # note: NOT .../api/ — that path 404s

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})

def cda_get(path, **params):
    r = session.get(f"{BASE}{path}", params=params, timeout=300)
    r.raise_for_status()
    return r.json()

def cda_post(path, body=None, **params):
    """POST a CDA query. Raises with the server's message on error."""
    r = session.post(f"{BASE}{path}", json=body or {}, params=params, timeout=300)
    if not r.ok:
        # CDA returns JSON errors; the WAF returns HTML (see the request-size section below)
        try:
            raise RuntimeError(f"HTTP {r.status_code}: {r.json()}")
        except ValueError:
            raise RuntimeError(f"HTTP {r.status_code} (non-JSON): {r.text[:200]}")
    return r.json()

# Every subject/file row carries four ontology-expansion columns per text field.
# They are useful for semantic search but make DataFrames unreadable.
TERM_SUFFIXES = ("_containing_terms", "_related_terms", "_slim_terms", "_synonym_terms")

def tidy(rows):
    """Rows -> DataFrame with the ontology-expansion columns dropped."""
    df = pd.DataFrame(rows)
    return df[[c for c in df.columns if not c.endswith(TERM_SUFFIXES)]]

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print("Interactive docs:", f"{BASE}/docs", "|", f"{BASE}/redoc")
print("OpenAPI spec:    ", f"{BASE}/openapi.json")

Interactive docs: https://cda.datacommons.cancer.gov/docs | https://cda.datacommons.cancer.gov/redoc
OpenAPI spec:     https://cda.datacommons.cancer.gov/openapi.json


## 1. The API surface

Seven endpoints, all unauthenticated. `GET` for the two metadata endpoints, `POST` (with a JSON body) for
everything else.

| Endpoint | Method | Purpose |
|---|---|---|
| `/columns/` | GET | Every queryable column, with table, type and description |
| `/release_metadata/` | GET | Per-node data versions, extraction dates, row counts |
| `/column_values/{column}` | POST | Distinct values and frequencies for one column |
| `/data/subject` | POST | Subject rows matching a filter |
| `/data/file` | POST | File rows matching a filter |
| `/summary/subject` | POST | Aggregate counts over matching subjects |
| `/summary/file` | POST | Aggregate counts over matching files |

The request body is the same shape for all `POST` endpoints:

```json
{
  "MATCH_ALL":        ["column OP value", ...],   // AND
  "MATCH_SOME":       ["column OP value", ...],   // OR
  "SEARCH_LIST":      ["free text", ...],         // ontology keyword search
  "ADD_COLUMNS":      ["column", ...],
  "EXCLUDE_COLUMNS":  ["column", ...],
  "COLLATE_RESULTS":  false,
  "EXTERNAL_REFERENCE": false
}
```

`/data/*` additionally take `limit` and `offset` as **query-string** parameters, not body fields.

In [3]:
spec = cda_get("/openapi.json")
for path, ops in spec["paths"].items():
    for method, op in ops.items():
        params = ", ".join(p["name"] for p in op.get("parameters", []))
        print(f"{method.upper():5s} {path:28s} {params}")

POST  /data/file                   limit, offset
POST  /data/subject                limit, offset
POST  /summary/file                
POST  /summary/subject             
POST  /column_values/{column}      column, data_source, limit, offset
GET   /release_metadata/           
GET   /columns/                    


## 2. Discovery — what can I query?

In [4]:
cols = pd.DataFrame(cda_get("/columns/")["result"])
print(f"{len(cols)} queryable columns across {cols['table'].nunique()} tables\n")
print(cols.groupby("table").size().sort_values(ascending=False).to_string())

105 queryable columns across 7 tables

table
mutation                33
file                    18
observation             17
subject                 14
project                 10
treatment               10
upstream_identifiers     3


In [5]:
# The linking table between IDC PatientIDs and CDA subjects
cols[cols["table"] == "upstream_identifiers"][["column", "data_type", "description"]]

,column,data_type,description
92,upstream_source,text,The upstream data source from which this identifier was extracted by CDA.
93,upstream_field,text,"What this identifier is called by the specified upstream data source, like for example ""case.sub..."
94,upstream_id,text,"One identifier, exactly as published by the specified upstream data source."


In [6]:
# Which upstream sources does CDA index, and how many identifiers from each?
pd.DataFrame(cda_post("/column_values/upstream_source")["result"])

,upstream_source,value_count
0,dbGaP,230
1,GC,263372
2,GDC,100779
3,ICDC,2812
4,IDC,256701
5,PDC,13740


In [7]:
# Current data release per node — how stale is each node's snapshot?
rel = pd.DataFrame(cda_get("/release_metadata/")["result"])
(rel.groupby("data_source")
    .agg(version=("data_source_version", "first"),
         extracted=("data_source_extraction_date", "max"),
         tables=("cda_table", "nunique"))
    .sort_index())

,version,extracted,tables
data_source,,,
CDA,June 2026,2026-06-25,6
GC,27.0,2026-05-14,5
GDC,"Data Release 45.0 - December 04, 2025",2026-05-22,6
ICDC,2026-04-16,2026-05-28,5
IDC,v24,2026-05-28,4
PDC,Data Release 6.1,2026-06-17,5


## 3. The raw response shape differs from `cdapython`

This is the one thing to internalise before porting notebook 02/03 code. The API returns **one boolean
column per CRDC node**:

```
subject_data_at_gc, subject_data_at_gdc, subject_data_at_icdc, subject_data_at_idc, subject_data_at_pdc
```

The `data_source: ['GDC', 'IDC']` list that notebooks 02 and 03 use is a **client-side reshape** performed
by `cdapython` — it is not a column in the API. Filtering on `data_source` over HTTP will fail.

In [8]:
resp = cda_post("/data/subject", {"MATCH_ALL": ["upstream_id = TCGA-DD-A4NO"]}, limit=1, offset=0)
row = resp["result"][0]

print("Response envelope keys:", list(resp.keys()))
print(f"total_row_count: {resp['total_row_count']}\n")
for k, v in row.items():
    if not k.endswith(TERM_SUFFIXES):
        print(f"  {k:28s} {v}")

Response envelope keys: ['result', 'query_sql', 'total_row_count', 'next_url']
total_row_count: 1

  subject_id                   TCGA.TCGA-DD-A4NO
  subject_crdc_id              None
  species                      human
  year_of_birth                None
  year_of_death                None
  cause_of_death               None
  race                         White
  ethnicity                    Non-Hispanic
  subject_data_at_gc           False
  subject_data_at_gdc          True
  subject_data_at_icdc         False
  subject_data_at_idc          True
  subject_data_at_pdc          False
  subject_data_source_count    2
  upstream_id                  ['75f78c38-6854-4390-ad11-789b4e549d63', 'e33806d9-19ad-48b4-a190-7af7b1d7da27', 'TCGA-DD-A4NO']


In [9]:
NODES = ["gc", "gdc", "icdc", "idc", "pdc"]

def data_source_list(row, prefix="subject"):
    """Reproduce cdapython's `data_source` column from the raw boolean flags."""
    return [n.upper() for n in NODES if row.get(f"{prefix}_data_at_{n}")]

print("reconstructed data_source:", data_source_list(row))
print("subject_id:               ", row["subject_id"])
print("upstream_id:              ", row["upstream_id"])

reconstructed data_source: ['GDC', 'IDC']
subject_id:                TCGA.TCGA-DD-A4NO
upstream_id:               ['75f78c38-6854-4390-ad11-789b4e549d63', 'e33806d9-19ad-48b4-a190-7af7b1d7da27', 'TCGA-DD-A4NO']


## 4. `query_sql` — every response shows its own SQL

Each `/data/*`, `/summary/*` and `/column_values/*` response carries the SQL CDA generated. This is the
fastest way to understand what a filter actually did — especially useful when a query returns 0 rows and
you need to know whether your column name, your value, or your assumption about the data model was wrong.

In [10]:
print(resp["query_sql"][:900], "...")

WITH filtered_preselect AS (SELECT subject.id_alias AS subject_id_alias FROM subject WHERE EXISTS (SELECT 1 FROM upstream_identifiers WHERE subject.id_alias = upstream_identifiers.id_alias AND upstream_identifiers.cda_table = :cda_table_1 AND coalesce(upper(upstream_identifiers.upstream_id), :coalesce_1) = upper(:upper_1))), upstream_identifiers_subject_columns AS (SELECT upstream_identifiers.id_alias AS id_alias, array_remove(array_agg(DISTINCT upstream_identifiers.upstream_id), NULL) AS upstream_id FROM upstream_identifiers WHERE upstream_identifiers.id_alias IN (SELECT filtered_preselect.subject_id_alias FROM filtered_preselect) AND upstream_identifiers.cda_table = :cda_table_1 GROUP BY upstream_identifiers.id_alias) SELECT row_to_json(json_subquery) AS json_results, (SELECT count(distinct(filtered_preselect.subject_id_alias)) AS count_1 FROM filtered_preselect) AS total_row_count FRO ...


## 5. Pagination — always pass `offset` explicitly

`/data/subject` and `/data/file` are paged. The response gives you `total_row_count` and a `next_url`.

**Gotcha:** `next_url` is built from the query parameters you sent. If you omit `offset` on the first
request, `next_url` comes back *without* an offset too — so following it re-fetches page 1 forever.
Always send `offset=0` on the first call, or compute offsets yourself as below.

In [11]:
def cda_rows(table, body, page_size=1000, max_rows=None):
    """Fetch all rows for a query, following pages by explicit offset."""
    rows, offset = [], 0
    while True:
        resp = cda_post(f"/data/{table}", body, limit=page_size, offset=offset)
        batch = resp["result"]
        rows.extend(batch)
        offset += len(batch)
        if not batch or offset >= resp["total_row_count"]:
            break
        if max_rows and len(rows) >= max_rows:
            break
    return rows[:max_rows] if max_rows else rows

# All files CDA knows about for this subject, across every node
files = cda_rows("file", {"MATCH_ALL": ["subject_id = TCGA.TCGA-DD-A4NO"]})
print(f"{len(files)} files")
tidy(files).head()

184 files


,file_id,file_crdc_id,file_name,file_description,drs_uri,access,size,format,file_type,category,file_data_at_gc,file_data_at_gdc,file_data_at_icdc,file_data_at_idc,file_data_at_pdc,file_data_source_count,anatomic_site,tumor_vs_normal,subject_id
0,01955bcc-d3e4-437c-868c-cee96e6bc755,None,f9a4aa72-57b8-4880-966e-a45d637d9767.wxs.aliquot_ensemble_raw.maf.gz,None,drs://dg.4dfc:01955bcc-d3e4-437c-868c-cee96e6bc755,controlled,116818,MAF,Aggregated Somatic Mutation,Simple Nucleotide Variation,False,True,False,False,False,1,[],"[normal, tumor]",[TCGA.TCGA-DD-A4NO]
1,0283defa-8d33-4775-a120-737d2a1cd840,None,402a5fcc-3fd0-4061-93e7-1565bdc84a99.rna_seq.star_splice_junctions.tsv.gz,None,drs://dg.4dfc:0283defa-8d33-4775-a120-737d2a1cd840,controlled,2187261,TSV,Splice Junction Quantification,Transcriptome Profiling,False,True,False,False,False,1,[],[tumor],[TCGA.TCGA-DD-A4NO]
2,03de9c4f-a6c8-4c56-9323-09ae2ef8d71f,None,402a5fcc-3fd0-4061-93e7-1565bdc84a99.rna_seq.chimeric.gdc_realn.bam,None,drs://dg.4dfc:03de9c4f-a6c8-4c56-9323-09ae2ef8d71f,controlled,94300562,BAM,Aligned Reads,Sequencing Reads,False,True,False,False,False,1,[],[tumor],[TCGA.TCGA-DD-A4NO]
3,06f6c284-6e54-4d25-b30c-92a913c17bf0,None,TCGA-LIHC.d471d7de-2e99-44e2-aa59-c571ff23ff9a.ascat2.allelic_specific.seg.txt,None,drs://dg.4dfc:06f6c284-6e54-4d25-b30c-92a913c17bf0,open,9530,TXT,Allele-specific Copy Number Segment,Copy Number Variation,False,True,False,False,False,1,[],"[normal, tumor]",[TCGA.TCGA-DD-A4NO]
4,0a50550b-db1b-4cf4-b047-146b1e932598,None,nationwidechildrens.org_clinical_omf_v4.0_lihc.txt,None,drs://dg.4dfc:0a50550b-db1b-4cf4-b047-146b1e932598,open,22037,BCR Biotab,Clinical Supplement,None,False,True,False,False,False,1,[],[],[TCGA.TCGA-DD-A4NO]


In [12]:
# What is available, by node and data category?
fdf = tidy(files)
fdf["node"] = fdf.apply(lambda r: ", ".join(data_source_list(r, prefix="file")), axis=1)

(fdf.groupby(["node", "category"])
    .agg(files=("file_id", "count"),
         total_size_MB=("size", lambda s: round(s.sum() / 1e6, 1)),
         access=("access", lambda s: "/".join(sorted(set(s.dropna())))))
    .reset_index())

,node,category,files,total_size_MB,access
0,GDC,Biospecimen,15,2113.0,open
1,GDC,Copy Number Variation,16,500.1,controlled/open
2,GDC,DNA Methylation,3,29.3,open
3,GDC,Proteome Profiling,1,0.0,open
4,GDC,Sequencing Reads,17,559686.2,controlled
5,GDC,Simple Nucleotide Variation,31,71.4,controlled/open
6,GDC,Somatic Structural Variation,5,0.3,controlled
7,GDC,Structural Variation,5,0.1,controlled
8,GDC,Transcriptome Profiling,4,6.7,controlled/open
9,IDC,Computed Tomography,18,937.9,open


## 6. Summaries — exact cross-node counts in a single call

`/summary/subject` returns a `data_source` object counting subjects by the **exact combination** of nodes
holding their data. The keys are underscore-joined node names with an `_exclusive` suffix, e.g.
`gdc_idc_exclusive` = "subjects present in GDC and IDC and nowhere else".

Because the buckets are mutually exclusive and exhaustive, you get complete cross-node coverage for an
arbitrary cohort from one request — no sampling, no per-patient loop.

In [13]:
def decode_data_source(ds):
    """Turn {'gdc_idc_exclusive': 448} into {('GDC','IDC'): 448}, dropping empty buckets.

    Note: one key in the API response (`gdc_pdc_icdc_gc_idc`, the all-five bucket) has no
    `_exclusive` suffix, so strip the suffix rather than splitting on it.
    """
    out = {}
    for key, count in ds.items():
        if not count:
            continue
        nodes = tuple(sorted(n.upper() for n in key.removesuffix("_exclusive").split("_")))
        out[nodes] = out.get(nodes, 0) + count
    return out

def node_totals(ds):
    """Subjects holding data at each node (buckets overlap here, by design)."""
    totals = {}
    for nodes, count in decode_data_source(ds).items():
        for n in nodes:
            totals[n] = totals.get(n, 0) + count
    return totals

### CDA indexes IDC collection IDs as `project_short_name`

This is the shortcut that makes exhaustive cross-node analysis cheap. An IDC `collection_id` such as
`tcga_lihc` is itself a CDA project, so `project_short_name = tcga_lihc` selects exactly the subjects
that IDC has imaging for in that collection — verified 1:1 against `idc-index` patient counts for all
46 TCGA and CPTAC collections (13,221 patients, zero mismatches) in
[notebook 03](03_cross_node_analysis.ipynb).

In [14]:
summary = cda_post("/summary/subject", {"MATCH_ALL": ["project_short_name = cptac_ccrcc"]})["result"][0]

print(f"subjects: {summary['total_count']}   files: {summary['file_count']}\n")
print("Exclusive node combinations:")
for nodes, n in sorted(decode_data_source(summary["data_source"]).items(), key=lambda kv: -kv[1]):
    print(f"  {' + '.join(nodes):24s} {n:5d}")
print("\nSubjects with data at each node:")
for node, n in sorted(node_totals(summary["data_source"]).items(), key=lambda kv: -kv[1]):
    print(f"  {node:6s} {n:5d}")

subjects: 233   files: 35365

Exclusive node combinations:
  GDC + IDC + PDC            121
  GC + GDC + IDC + PDC       110
  IDC                          1
  GDC + IDC                    1

Subjects with data at each node:
  IDC      233
  GDC      232
  PDC      231
  GC       110


**Scope caveat.** `/summary/file` filtered the same way counts only that project's *own* files. For
`tcga_lihc` that is 3,457 DICOM files, all `idc_exclusive` — the genomic files for those same subjects
live under the GDC project. To reach files across nodes you must go through `subject_id`, as in section 5.

In [15]:
fsum = cda_post("/summary/file", {"MATCH_ALL": ["project_short_name = tcga_lihc"]})["result"][0]
print(f"files: {fsum['total_count']}  subjects: {fsum['subject_count']}")
print("by node:", {" + ".join(k): v for k, v in decode_data_source(fsum["data_source"]).items()})
print("\nby category:")
for c in fsum["category_summary"]:
    print(f"  {c['category']:32s} {c['count_result']:6d}")

files: 3457  subjects: 377
by node: {'IDC': 3457}

by category:
  Positron emission tomography          1
  Computed Tomography                 777
  Slide Microscopy                    870
  Magnetic Resonance                  910
  Segmentation                        899


## 7. `ADD_COLUMNS` — pull in fields from other tables

Subject queries return only `subject` columns by default. `ADD_COLUMNS` joins fields from `observation`,
`project` and the other tables onto the result. Added columns arrive as **lists**, because a subject can
have several diagnoses, stages or project memberships.

In [16]:
enriched = cda_post(
    "/data/subject",
    {"MATCH_ALL": ["upstream_id = TCGA-DD-A4NO"],
     "ADD_COLUMNS": ["project_short_name", "diagnosis", "stage", "sex", "vital_status"]},
    limit=1, offset=0,
)["result"][0]

for k in ["project_short_name", "diagnosis", "stage", "sex", "vital_status"]:
    print(f"{k:20s} {enriched[k]}")

project_short_name   ['phs000178', 'tcga', 'TCGA', 'tcga_lihc', 'TCGA-LIHC']
diagnosis            ['Hepatocellular carcinoma']
stage                ['Stage I']
sex                  ['male']
vital_status         ['alive']


Note what `project_short_name` contains: `['phs000178', 'tcga', 'TCGA', 'tcga_lihc', 'TCGA-LIHC']` — the
dbGaP accession, the program in two casings, the **IDC collection ID**, and the GDC project. All five are
valid filter values, which is exactly why `project_short_name = tcga_lihc` works in section 6.

**Gotcha:** `EXCLUDE_COLUMNS` only accepts columns listed by `/columns/`. The `*_containing_terms` /
`*_related_terms` / `*_slim_terms` / `*_synonym_terms` fields are *not* listed there, so you cannot exclude
them server-side — they always come back, and you drop them client-side (that is what `tidy()` does).

In [17]:
try:
    cda_post("/data/subject",
             {"MATCH_ALL": ["upstream_id = TCGA-DD-A4NO"],
              "EXCLUDE_COLUMNS": ["species_containing_terms"]},
             limit=1, offset=0)
except RuntimeError as e:
    print(e)

HTTP 400: {'error_type': 'ColumnNotFound', 'message': 'Column Not Found: species_containing_terms'}


## 8. `SEARCH_LIST` — ontology keyword search (REST-only)

`SEARCH_LIST` matches free text against a keyword index built over the ontology expansions of every text
field, spanning tables. It is in the live OpenAPI spec but **not** in `cda_client` 2.0.1's
`DataRequestBody`, so it is unreachable from `cdapython` as pinned in this repo.

In [18]:
gbm = cda_post("/data/subject", {"SEARCH_LIST": ["glioblastoma"]}, limit=5, offset=0)
print(f"subjects matching 'glioblastoma': {gbm['total_row_count']:,}\n")
tidy(gbm["result"])[["subject_id", "subject_data_at_idc", "subject_data_at_gdc",
                     "subject_data_at_pdc", "subject_data_source_count"]]

subjects matching 'glioblastoma': 2,283



,subject_id,subject_data_at_idc,subject_data_at_gdc,subject_data_at_pdc,subject_data_source_count
0,CPTAC.C3L-03728,True,True,True,4
1,Cancer Proteogenomics Group of National Cancer Center Korea.KNCC_GBM0016,False,False,True,1
2,FM.AD15309,False,True,False,1
3,TCGA.TCGA-06-0241,True,True,False,3
4,HCMI.HCM-BROD-0198-C71,True,True,True,3


In [19]:
# How many of those also have imaging in IDC? One summary call.
gbm_sum = cda_post("/summary/subject", {"SEARCH_LIST": ["glioblastoma"]})["result"][0]
totals = node_totals(gbm_sum["data_source"])
print(f"total: {gbm_sum['total_count']:,}")
for node, n in sorted(totals.items(), key=lambda kv: -kv[1]):
    print(f"  {node:6s} {n:6,d}  ({n / gbm_sum['total_count'] * 100:.0f}%)")

total: 2,283
  IDC     1,519  (67%)
  GDC     1,459  (64%)
  GC        426  (19%)
  PDC       345  (15%)


## 9. `EXTERNAL_REFERENCE` — pointers to derived data outside CRDC

Set `EXTERNAL_REFERENCE: true` and subject rows gain an `external_reference_columns` field listing
related resources — chiefly ISB-CGC BigQuery tables of derived data (methylation, expression, clinical)
for the subject's program.

In [20]:
ext = cda_post("/data/subject",
               {"MATCH_ALL": ["upstream_id = TCGA-DD-A4NO"], "EXTERNAL_REFERENCE": True},
               limit=1, offset=0)["result"][0]["external_reference_columns"]

print(f"{len(ext)} external references\n")
pd.DataFrame(ext)[["external_reference_name", "external_reference_short_name",
                   "source_short_name", "last_updated"]].head(10)

8 external references



,external_reference_name,external_reference_short_name,source_short_name,last_updated
0,TCGA HG19 DNA METHYLATION,isb-cgc-bq.TCGA.DNA_methylation_hg19_gdc_current,ISB-CGC,2024-10-16
1,TCGA HG38 DNA METHYLATION,isb-cgc-bq.TCGA.DNA_methylation_hg38_gdc_current,ISB-CGC,2024-10-16
2,TCGA HG38 COPY NUMBER VARIATION GENE LEVEL,isb-cgc-bq.TCGA.copy_number_gene_level_hg38_gdc_current,ISB-CGC,2025-11-12
3,TCGA HG38 COPY NUMBER SEGMENT ALLELIC,isb-cgc-bq.TCGA.copy_number_segment_allelic_hg38_gdc_current,ISB-CGC,2024-10-16
4,TCGA HG19 COPY NUMBER SEGMENT MASKED,isb-cgc-bq.TCGA.copy_number_segment_masked_hg19_gdc_current,ISB-CGC,2024-10-16
5,TCGA HG38 COPY NUMBER SEGMENT MASKED,isb-cgc-bq.TCGA.copy_number_segment_masked_hg38_gdc_current,ISB-CGC,2024-10-16
6,TCGA HG38 SOMATIC MUTATION,isb-cgc-bq.TCGA.masked_somatic_mutation_hg38_gdc_current,ISB-CGC,2024-10-16
7,TCGA HG19 PROTEIN EXPRESSION,isb-cgc-bq.TCGA.protein_expression_hg19_gdc_current,ISB-CGC,2024-10-16


## 10. Request-size limit: bodies over ~8 KB are rejected

A gateway in front of the API rejects large request bodies with a bare **HTTP 403 and an HTML page** — not
a JSON error, and nothing in the OpenAPI spec mentions it. Measured on this API:

| `MATCH_SOME` filters | body size | result |
|---|---|---|
| 200 | 6.0 KB | 200 OK |
| 260 | 7.8 KB | 200 OK |
| 300 | 9.0 KB | **403** |

This applies to `cdapython` too — same API, same gateway. If you batch patient IDs into `MATCH_SOME`,
chunk at ~200 filters. (For IDC collections you usually don't need to: `project_short_name` from section 6
selects a whole collection with one short filter.)

In [21]:
import json as _json

def probe(n_filters):
    body = {"MATCH_SOME": [f"upstream_id = TCGA-{i:02d}-{i:04d}" for i in range(n_filters)]}
    size = len(_json.dumps(body))
    try:
        cda_post("/summary/subject", body)
        return size, "200 OK"
    except RuntimeError as e:
        return size, str(e)[:60]

for n in (200, 300):
    size, outcome = probe(n)
    print(f"{n:4d} filters  {size/1024:5.1f} KB  ->  {outcome}")

 200 filters    6.0 KB  ->  200 OK
 300 filters    9.0 KB  ->  HTTP 403 (non-JSON): <html>
<head><title>403 Forbidden</tit


In [22]:
def chunked_match_some(values, column="upstream_id", chunk_size=200):
    """Yield MATCH_SOME bodies small enough to clear the ~8 KB gateway limit."""
    filters = [f"{column} = {v}" for v in values]
    for i in range(0, len(filters), chunk_size):
        yield {"MATCH_SOME": filters[i:i + chunk_size]}

def summarize_in_chunks(values, column="upstream_id", chunk_size=200):
    """Exact node totals for an arbitrary ID list. Buckets are disjoint, so they sum across chunks."""
    matched, totals = 0, {}
    for body in chunked_match_some(values, column, chunk_size):
        s = cda_post("/summary/subject", body)["result"][0]
        matched += s["total_count"]
        for node, n in node_totals(s["data_source"]).items():
            totals[node] = totals.get(node, 0) + n
    return matched, totals

# Demo on a small list of TCGA barcodes
ids = ["TCGA-DD-A4NO", "TCGA-CV-7183", "TCGA-OR-A5J1", "C3N-02973", "NOT-A-REAL-ID"]
matched, totals = summarize_in_chunks(ids)
print(f"matched {matched}/{len(ids)} subjects in CDA")
print("node totals:", totals)

matched 4/5 subjects in CDA
node totals: {'GDC': 4, 'IDC': 4, 'PDC': 1, 'GC': 1}


## 11. The same query from other languages

Nothing above is Python-specific. The core cross-node lookup in `curl`:

```bash
curl -s -X POST https://cda.datacommons.cancer.gov/data/subject \
  -H 'Content-Type: application/json' \
  -d '{"MATCH_ALL":["upstream_id = TCGA-DD-A4NO"]}' \
  | jq '.result[0] | {subject_id, subject_data_at_gdc, subject_data_at_idc, subject_data_at_pdc}'
```

Exact cross-node counts for an IDC collection:

```bash
curl -s -X POST https://cda.datacommons.cancer.gov/summary/subject \
  -H 'Content-Type: application/json' \
  -d '{"MATCH_ALL":["project_short_name = cptac_ccrcc"]}' \
  | jq '.result[0] | {total_count, file_count,
                      nodes: (.data_source | with_entries(select(.value > 0)))}'
```

And in R:

```r
library(httr2)
library(jsonlite)

resp <- request("https://cda.datacommons.cancer.gov/summary/subject") |>
  req_body_json(list(MATCH_ALL = list("project_short_name = cptac_ccrcc"))) |>
  req_perform() |>
  resp_body_json()

ds <- resp$result[[1]]$data_source
Filter(function(x) x > 0, ds)
```

## Summary — which interface should you use?

| | REST (this notebook) | `cdapython` (notebooks 02, 03) |
|---|---|---|
| Install | `requests` only | `cdapython` + `cda-client` + pinned pandas |
| Language | any | Python |
| Returns | JSON dicts | DataFrames |
| Pagination | you loop (~10 lines) | automatic |
| Column validation | server-side, after the round trip | client-side, before |
| `data_source` as a list | reconstruct from booleans | provided |
| Generated SQL (`query_sql`) | yes | not surfaced |
| `SEARCH_LIST` keyword search | yes | not in `cda_client` 2.0.1 |
| Batch-from-CSV helper | write your own | `match_from_file=` |
| ~8 KB request limit | applies | applies |

**Use `cdapython`** for interactive analysis in Python where you want DataFrames immediately.
**Use REST** when you want no dependencies, another language, ontology keyword search, or the generated SQL
— and to stay insulated from client-version churn.

Both hit the same service and the same data. Nothing in this notebook required authentication.

### Further reading

- Swagger UI — <https://cda.datacommons.cancer.gov/docs>
- ReDoc — <https://cda.datacommons.cancer.gov/redoc>
- API source — <https://github.com/CancerDataAggregator/cda-api>
- [developer/cda_overview.md](../developer/cda_overview.md) — data model and column reference
- [developer/idc_cda_integration.md](../developer/idc_cda_integration.md) — IDC ↔ CDA identifier mapping